# Aufgaben 20.07.2026

## Python mit Erklärungen

Enthalten sind:

1. Disko-Türsteher
2. Parkhaus-Statusanzeige
3. Logikgatter und Wahrheitstabellen
4. Multiplikationstabellen
5. Sitzplan mit verschachtelten Listen
6. Weitere Auswertungen des Sitzplans
7. List Comprehensions
8. Lottozahlengenerator „6 aus 49“
9. Figuren mit Schleifen
10. Geldbetrag in Scheine und Münzen zerlegen

---

> **Hinweis:** Die Codezellen können nacheinander ausgeführt werden. Die meisten Beispiele verwenden feste Werte, damit die Ergebnisse nachvollziehbar bleiben.

# 1. Disko-Türsteher

## Aufgabenstellung

Vor einer Diskothek sitzt ein Roboter. Seine Sensoren erfassen die Eigenschaften eines Gastes. Das Laufband beziehungsweise die Anzeige des Roboters gibt anschließend eine Entscheidung als Text aus.

Regeln:

- Personen unter 18 Jahren dürfen nur bis 00:00 Uhr in die Diskothek.
- VIP-Gäste erhalten ein Glas Sekt.
- Personen mit einem gelben T-Shirt erhalten ebenfalls ein Glas Sekt.
- Personen unter 18 Jahren dürfen keinen Alkohol erhalten.
- Jede Person erhält höchstens ein Glas Sekt.
- Alle anderen Gäste dürfen normal hinein.

Die Eingaben sollen zunächst über feste Variablen erfolgen. Die Ausgabe erfolgt mit `print()`.

## Überlegung

Wir müssen zwei Entscheidungen voneinander trennen:

1. **Darf die Person hinein?**
2. **Bekommt sie Sekt?**

Wichtig ist die Reihenfolge:

- Zuerst wird geprüft, ob eine minderjährige Person nach Mitternacht kommt.
- Erst wenn der Einlass erlaubt ist, wird über den Sekt entschieden.
- Minderjährige bekommen grundsätzlich keinen Alkohol.
- VIP und gelbes T-Shirt werden mit `or` verbunden. Dadurch bleibt es bei höchstens einem Glas.

In [ ]:
# Eingangsparameter des Roboters
alter = 25
uhrzeit = 23
ist_vip = False
shirtfarbe = "gelb"

# Zuerst wird der Einlass geprüft.
if alter < 18 and uhrzeit >= 24:
    print("Kein Einlass: Personen unter 18 Jahren dürfen nach 00:00 Uhr nicht hinein.")
else:
    print("Einlass erlaubt.")

    # Alkohol darf nur an volljährige Personen ausgegeben werden.
    if alter >= 18 and (ist_vip or shirtfarbe.lower() == "gelb"):
        print("Der Gast erhält ein Glas Sekt.")
    elif alter < 18:
        print("Der Gast erhält keinen Alkohol.")
    else:
        print("Der Gast kommt ganz normal hinein und erhält keinen Sekt.")

Einlass erlaubt.
Der Gast erhält ein Glas Sekt.


### Warum funktioniert die Bedingung?

```python
alter >= 18 and (ist_vip or shirtfarbe.lower() == "gelb")
```

Der Sekt wird nur ausgegeben, wenn **beide Hauptbedingungen** erfüllt sind:

- Die Person ist mindestens 18 Jahre alt.
- Die Person ist VIP **oder** trägt ein gelbes T-Shirt.

Durch die gemeinsame Bedingung wird nur ein `print()` für den Sekt ausgeführt. Ein VIP mit gelbem T-Shirt bekommt daher nicht zwei Gläser.

> **Merksatz:** Mit Klammern wird deutlich festgelegt, welche Teilbedingungen zusammengehören.

## Erweiterte und wiederverwendbare Lösung als Funktion

Eine Funktion ist sinnvoll, wenn der Roboter viele Gäste nacheinander prüfen soll.

In [ ]:
def pruefe_einlass(alter, uhrzeit, ist_vip, shirtfarbe):
    """Prüft Einlass und Sektausgabe für einen Gast."""

    if alter < 0:
        return "Fehler: Das Alter darf nicht negativ sein."

    if not 0 <= uhrzeit <= 24:
        return "Fehler: Die Uhrzeit muss zwischen 0 und 24 liegen."

    if alter < 18 and uhrzeit >= 24:
        return "Kein Einlass: minderjährig und nach 00:00 Uhr."

    if alter >= 18 and (ist_vip or shirtfarbe.lower() == "gelb"):
        return "Einlass erlaubt. Der Gast erhält ein Glas Sekt."

    if alter < 18:
        return "Einlass erlaubt. Der Gast erhält keinen Alkohol."

    return "Einlass erlaubt. Der Gast kommt normal hinein."


# Mehrere Testfälle
testgaeste = [
    (17, 23, False, "blau"),
    (17, 24, True, "gelb"),
    (25, 22, True, "schwarz"),
    (30, 21, False, "gelb"),
    (40, 23, False, "rot"),
]

for gast in testgaeste:
    print(gast, "->", pruefe_einlass(*gast))

(17, 23, False, 'blau') -> Einlass erlaubt. Der Gast erhält keinen Alkohol.
(17, 24, True, 'gelb') -> Kein Einlass: minderjährig und nach 00:00 Uhr.
(25, 22, True, 'schwarz') -> Einlass erlaubt. Der Gast erhält ein Glas Sekt.
(30, 21, False, 'gelb') -> Einlass erlaubt. Der Gast erhält ein Glas Sekt.
(40, 23, False, 'rot') -> Einlass erlaubt. Der Gast kommt normal hinein.


> **Prüfungsblick:** Bei verschachtelten Bedingungen sollte zuerst die Regel geprüft werden, die einen weiteren Ablauf vollständig verhindert. Hier ist das der verweigerte Einlass.

# 2. Parkhaus-Statusanzeige

## Aufgabenstellung

Ein Parkhaus besitzt insgesamt **500 Stellplätze**. Die Anzahl der belegten Stellplätze wird automatisch ermittelt.

Zusätzliche Bedingungen:

- Die Ein- oder Ausfahrt kann blockiert sein.
- Das Parkhaus kann für eine Veranstaltung angemietet sein.
- Bei einer Auslastung von über 95 Prozent soll „Parkhaus belegt“ angezeigt werden.
- Ist eine Ein- oder Ausfahrt blockiert oder wurde das Parkhaus angemietet, soll „Parkhaus gesperrt“ angezeigt werden.
- Andernfalls soll „Parkhaus frei“ angezeigt werden.

## Auswertung der Entscheidungstabelle

Die Tabelle lässt sich zu drei Regeln vereinfachen:

1. **Blockiert oder angemietet:** Parkhaus gesperrt
2. **Nicht gesperrt und über 95 Prozent ausgelastet:** Parkhaus belegt
3. **Keine der vorherigen Bedingungen:** Parkhaus frei

Die Sperrung besitzt Vorrang. Auch bei geringer Auslastung ist das Parkhaus gesperrt, sobald eine Zufahrt blockiert oder das Parkhaus angemietet ist.

In [ ]:
def status_anzeigen(
    gesamtplaetze,
    belegte_plaetze,
    zufahrt_blockiert,
    fuer_veranstaltung_angemietet
):
    """Ermittelt den Status eines Parkhauses anhand der Entscheidungstabelle."""

    # Eingaben überprüfen
    if gesamtplaetze <= 0:
        print("Fehler: Die Anzahl der Gesamtplätze muss größer als 0 sein.")
        return

    if belegte_plaetze < 0 or belegte_plaetze > gesamtplaetze:
        print("Fehler: Die belegten Plätze liegen außerhalb des gültigen Bereichs.")
        return

    auslastung = belegte_plaetze / gesamtplaetze * 100

    # Sperrung hat laut Entscheidungstabelle Vorrang.
    if zufahrt_blockiert or fuer_veranstaltung_angemietet:
        status = "Parkhaus gesperrt"
    elif auslastung > 95:
        status = "Parkhaus belegt"
    else:
        status = "Parkhaus frei"

    print(f"Belegte Stellplätze: {belegte_plaetze} von {gesamtplaetze}")
    print(f"Auslastung: {auslastung:.1f} %")
    print(f"Status: {status}")


# Beispiel aus der Aufgabenstellung
status_anzeigen(500, 330, True, False)

Belegte Stellplätze: 330 von 500
Auslastung: 66.0 %
Status: Parkhaus gesperrt


## Weitere Testfälle

Die Grenze „über 95 Prozent“ ist wichtig:

- Genau 95 Prozent sind **nicht über** 95 Prozent.
- Bei 500 Plätzen entsprechen 95 Prozent genau 475 belegten Plätzen.
- Erst ab 476 belegten Plätzen ist die Auslastung größer als 95 Prozent.

In [ ]:
testfaelle = [
    (500, 475, False, False),  # genau 95 %
    (500, 476, False, False),  # über 95 %
    (500, 100, True, False),   # blockiert
    (500, 100, False, True),   # angemietet
    (500, 500, False, True),   # voll und angemietet
]

for testfall in testfaelle:
    print("\n--- Testfall", testfall, "---")
    status_anzeigen(*testfall)


--- Testfall (500, 475, False, False) ---
Belegte Stellplätze: 475 von 500
Auslastung: 95.0 %
Status: Parkhaus frei

--- Testfall (500, 476, False, False) ---
Belegte Stellplätze: 476 von 500
Auslastung: 95.2 %
Status: Parkhaus belegt

--- Testfall (500, 100, True, False) ---
Belegte Stellplätze: 100 von 500
Auslastung: 20.0 %
Status: Parkhaus gesperrt

--- Testfall (500, 100, False, True) ---
Belegte Stellplätze: 100 von 500
Auslastung: 20.0 %
Status: Parkhaus gesperrt

--- Testfall (500, 500, False, True) ---
Belegte Stellplätze: 500 von 500
Auslastung: 100.0 %
Status: Parkhaus gesperrt


> **Merksatz:** Eine Entscheidungstabelle hilft dabei, viele Kombinationen systematisch zu prüfen. Anschließend können Regeln mit gleicher Aktion häufig zusammengefasst werden.

# 3. Logikgatter und Wahrheitstabellen

## Grundlagen

In Python werden logische Zustände mit `False` und `True` dargestellt.

| Logikgatter | Bedeutung | Python |
|---|---|---|
| NOT | Negation | `not A` |
| AND | Beide Eingänge müssen wahr sein | `A and B` |
| NAND | Negiertes AND | `not (A and B)` |
| OR | Mindestens ein Eingang ist wahr | `A or B` |
| NOR | Negiertes OR | `not (A or B)` |
| XOR | Genau ein Eingang ist wahr | `A != B` |
| XNOR | Beide Eingänge sind gleich | `A == B` |

Für Wahrheitstabellen werden alle möglichen Kombinationen der Eingangswerte geprüft.

In [ ]:
werte = [False, True]

print("A | B | NOT A | AND | NAND | OR | NOR | XOR | XNOR")
print("-" * 57)

for a in werte:
    for b in werte:
        print(
            f"{int(a)} | {int(b)} |"
            f"   {int(not a)}   |"
            f"  {int(a and b)}  |"
            f"   {int(not (a and b))}  |"
            f" {int(a or b)}  |"
            f"  {int(not (a or b))}  |"
            f"  {int(a != b)}  |"
            f"   {int(a == b)}"
        )

A | B | NOT A | AND | NAND | OR | NOR | XOR | XNOR
---------------------------------------------------------
0 | 0 |   1   |  0  |   1  | 0  |  1  |  0  |   1
0 | 1 |   1   |  0  |   1  | 1  |  0  |  1  |   0
1 | 0 |   0   |  0  |   1  | 1  |  0  |  1  |   0
1 | 1 |   0   |  1  |   0  | 1  |  0  |  0  |   1


## Aufgabe 1 – Herleitung der Schaltung

Die obere UND-Verknüpfung erhält:

- den negierten Eingang `x`
- den normalen Eingang `y`

Daraus entsteht:

```text
nicht x UND y
```

Die untere UND-Verknüpfung erhält:

- den normalen Eingang `x`
- den negierten Eingang `y`

Daraus entsteht:

```text
x UND nicht y
```

Beide Zwischenergebnisse werden durch ein ODER-Gatter verbunden:

```text
(nicht x UND y) ODER (x UND nicht y)
```

Diese Formel ist genau dann wahr, wenn sich `x` und `y` unterscheiden. Das entspricht einem **XOR-Gatter**.

In [ ]:
def schaltung_1(x, y):
    oberer_zweig = (not x) and y
    unterer_zweig = x and (not y)
    s = oberer_zweig or unterer_zweig
    return oberer_zweig, unterer_zweig, s


print("x | y | oberer Zweig | unterer Zweig | s")
print("-" * 45)

for x in werte:
    for y in werte:
        oben, unten, s = schaltung_1(x, y)
        print(
            f"{int(x)} | {int(y)} |"
            f"       {int(oben)}       |"
            f"        {int(unten)}       | {int(s)}"
        )

x | y | oberer Zweig | unterer Zweig | s
---------------------------------------------
0 | 0 |       0       |        0       | 0
0 | 1 |       1       |        0       | 1
1 | 0 |       0       |        1       | 1
1 | 1 |       0       |        0       | 0


### Wahrheitstabelle der ersten Schaltung

| x | y | s |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

**Ableitung:** Die Schaltung bildet ein XOR-Gatter nach.

## Aufgabe 2 – XOR nur mit NAND-Gattern

Die Kreise an den Ausgängen der UND-Gatter zeigen eine Negation. Es handelt sich daher um NAND-Gatter.

Die Schaltung kann schrittweise beschrieben werden:

```text
n1 = NAND(x, y)
n2 = NAND(x, n1)
n3 = NAND(n1, y)
s  = NAND(n2, n3)
```

Diese bekannte NAND-Schaltung erzeugt ebenfalls XOR.

In [ ]:
def nand(a, b):
    return not (a and b)


def schaltung_2(x, y):
    n1 = nand(x, y)
    n2 = nand(x, n1)
    n3 = nand(n1, y)
    s = nand(n2, n3)
    return n1, n2, n3, s


print("x | y | n1 | n2 | n3 | s")
print("-" * 29)

for x in werte:
    for y in werte:
        n1, n2, n3, s = schaltung_2(x, y)
        print(
            f"{int(x)} | {int(y)} |"
            f" {int(n1)}  | {int(n2)}  | {int(n3)}  | {int(s)}"
        )

x | y | n1 | n2 | n3 | s
-----------------------------
0 | 0 | 1  | 1  | 1  | 0
0 | 1 | 1  | 1  | 0  | 1
1 | 0 | 1  | 0  | 1  | 1
1 | 1 | 0  | 1  | 1  | 0


### Wahrheitstabelle der zweiten Schaltung

| x | y | s |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

## Gesamtergebnis

Beide Schaltungen besitzen dieselbe Wahrheitstabelle und sind daher **logisch äquivalent**.

```text
Schaltung 1 = XOR
Schaltung 2 = XOR
```

> **Prüfungsblick:** Zwei unterschiedlich aufgebaute Schaltungen sind logisch gleichwertig, wenn ihre Ausgänge für jede mögliche Eingangskombination übereinstimmen.

# 4. Multiplikationstabellen

## Aufgabenstellung

Erzeuge die Multiplikationstabellen von 1 bis 10.

## Lösungsweg

Wir benötigen zwei Schleifen:

- Die äußere Schleife bestimmt die jeweilige Zahl beziehungsweise Tabelle.
- Die innere Schleife multipliziert diese Zahl nacheinander mit 1 bis 10.

Da eine Schleife innerhalb einer anderen Schleife steht, spricht man von **verschachtelten Schleifen**.

In [ ]:
for zahl in range(1, 11):
    print(f"\n--- {zahl}er-Reihe ---")

    for faktor in range(1, 11):
        ergebnis = zahl * faktor
        print(f"{zahl:2} × {faktor:2} = {ergebnis:3}")


--- 1er-Reihe ---
 1 ×  1 =   1
 1 ×  2 =   2
 1 ×  3 =   3
 1 ×  4 =   4
 1 ×  5 =   5
 1 ×  6 =   6
 1 ×  7 =   7
 1 ×  8 =   8
 1 ×  9 =   9
 1 × 10 =  10

--- 2er-Reihe ---
 2 ×  1 =   2
 2 ×  2 =   4
 2 ×  3 =   6
 2 ×  4 =   8
 2 ×  5 =  10
 2 ×  6 =  12
 2 ×  7 =  14
 2 ×  8 =  16
 2 ×  9 =  18
 2 × 10 =  20

--- 3er-Reihe ---
 3 ×  1 =   3
 3 ×  2 =   6
 3 ×  3 =   9
 3 ×  4 =  12
 3 ×  5 =  15
 3 ×  6 =  18
 3 ×  7 =  21
 3 ×  8 =  24
 3 ×  9 =  27
 3 × 10 =  30

--- 4er-Reihe ---
 4 ×  1 =   4
 4 ×  2 =   8
 4 ×  3 =  12
 4 ×  4 =  16
 4 ×  5 =  20
 4 ×  6 =  24
 4 ×  7 =  28
 4 ×  8 =  32
 4 ×  9 =  36
 4 × 10 =  40

--- 5er-Reihe ---
 5 ×  1 =   5
 5 ×  2 =  10
 5 ×  3 =  15
 5 ×  4 =  20
 5 ×  5 =  25
 5 ×  6 =  30
 5 ×  7 =  35
 5 ×  8 =  40
 5 ×  9 =  45
 5 × 10 =  50

--- 6er-Reihe ---
 6 ×  1 =   6
 6 ×  2 =  12
 6 ×  3 =  18
 6 ×  4 =  24
 6 ×  5 =  30
 6 ×  6 =  36
 6 ×  7 =  42
 6 ×  8 =  48
 6 ×  9 =  54
 6 × 10 =  60

--- 7er-Reihe ---
 7 ×  1 =   7
 7 ×  2 =  14

### Erklärung zu `range(1, 11)`

Der Endwert von `range()` gehört nicht mehr zum erzeugten Bereich. Deshalb erzeugt `range(1, 11)` die Zahlen von 1 bis einschließlich 10.

Die Angaben `:2` und `:3` sorgen für eine rechtsbündige Ausgabe mit einer festen Mindestbreite.

> **Merksatz:** Bei verschachtelten Schleifen wird die innere Schleife für jeden einzelnen Durchlauf der äußeren Schleife vollständig ausgeführt.

# 5. Sitzplan mit verschachtelten Listen

## Aufgabenstellung

Eine Klasse besitzt vier Reihen mit jeweils fünf Sitzplätzen. Die Namen werden in einer verschachtelten Liste gespeichert.

Gesucht sind:

1. alle Namen aus Reihe 2,
2. der dritte Sitzplatz aus Reihe 3,
3. der erste Sitz jeder Reihe,
4. das Ersetzen eines bestimmten Schülers durch `"Frei"`.

## Ausgangsdaten

Python zählt Listenindizes ab 0:

- Reihe 1 besitzt den Index `0`.
- Reihe 2 besitzt den Index `1`.
- Der dritte Platz besitzt den Index `2`.

In [ ]:
sitzplan = [
    ["Anna", "Ben", "Cem", "Daria", "Elias"],
    ["Fatma", "Gina", "Hannes", "Ida", "Jonas"],
    ["Kira", "Leon", "Mert", "Nina", "Oskar"],
    ["Pia", "Quentin", "Rita", "Sven", "Tina"],
]

print("Kompletter Sitzplan:")
for reihennummer, reihe in enumerate(sitzplan, start=1):
    print(f"Reihe {reihennummer}: {reihe}")

Kompletter Sitzplan:
Reihe 1: ['Anna', 'Ben', 'Cem', 'Daria', 'Elias']
Reihe 2: ['Fatma', 'Gina', 'Hannes', 'Ida', 'Jonas']
Reihe 3: ['Kira', 'Leon', 'Mert', 'Nina', 'Oskar']
Reihe 4: ['Pia', 'Quentin', 'Rita', 'Sven', 'Tina']


In [ ]:
# 1. Alle Namen aus Reihe 2
print("Reihe 2:", sitzplan[1])

# 2. Dritter Sitzplatz aus Reihe 3
print("Dritter Sitzplatz in Reihe 3:", sitzplan[2][2])

# 3. Erster Sitz jeder Reihe
print("\nErster Sitz jeder Reihe:")
for reihe in sitzplan:
    print(reihe[0])

# 4. Nina durch 'Frei' ersetzen
for reihe in sitzplan:
    for platz_index, name in enumerate(reihe):
        if name == "Nina":
            reihe[platz_index] = "Frei"

print("\nSitzplan nach der Änderung:")
for reihennummer, reihe in enumerate(sitzplan, start=1):
    print(f"Reihe {reihennummer}: {reihe}")

Reihe 2: ['Fatma', 'Gina', 'Hannes', 'Ida', 'Jonas']
Dritter Sitzplatz in Reihe 3: Mert

Erster Sitz jeder Reihe:
Anna
Fatma
Kira
Pia

Sitzplan nach der Änderung:
Reihe 1: ['Anna', 'Ben', 'Cem', 'Daria', 'Elias']
Reihe 2: ['Fatma', 'Gina', 'Hannes', 'Ida', 'Jonas']
Reihe 3: ['Kira', 'Leon', 'Mert', 'Frei', 'Oskar']
Reihe 4: ['Pia', 'Quentin', 'Rita', 'Sven', 'Tina']


## Zugriff auf verschachtelte Listen

```python
sitzplan[2][2]
```

bedeutet:

1. `sitzplan[2]` wählt die dritte Reihe.
2. Das zweite `[2]` wählt daraus den dritten Sitzplatz.

> **Merksatz:** Bei einer verschachtelten Liste wird zuerst die äußere und danach die innere Position angegeben.

# 6. Weitere Aufgaben zur Schülerliste

Gesucht sind:

- alle Namen, die mit `A` beginnen,
- die Gesamtzahl der Schüler beziehungsweise belegten Einträge,
- eine flache Liste mit allen Namen,
- die Position eines bestimmten Schülers.

In [ ]:
# Für diese Auswertungen wird ein unveränderter Sitzplan verwendet.
sitzplan_auswertung = [
    ["Anna", "Ben", "Cem", "Daria", "Elias"],
    ["Amir", "Gina", "Hannes", "Ida", "Jonas"],
    ["Kira", "Leon", "Mert", "Nina", "Oskar"],
    ["Pia", "Quentin", "Rita", "Sven", "Tina"],
]

# 1. Alle Namen, die mit A beginnen
namen_mit_a = []

for reihe in sitzplan_auswertung:
    for name in reihe:
        if name.startswith("A"):
            namen_mit_a.append(name)

print("Namen mit A:", namen_mit_a)

# 2. Gesamtzahl aller Einträge
anzahl_schueler = 0

for reihe in sitzplan_auswertung:
    anzahl_schueler += len(reihe)

print("Anzahl der Schüler:", anzahl_schueler)

# 3. Flache Liste erzeugen
alle_namen = []

for reihe in sitzplan_auswertung:
    for name in reihe:
        alle_namen.append(name)

print("Flache Liste:", alle_namen)

# 4. Bestimmten Schüler suchen
gesuchter_name = "Mert"
position = None

for reihe_index, reihe in enumerate(sitzplan_auswertung):
    for platz_index, name in enumerate(reihe):
        if name == gesuchter_name:
            position = (reihe_index, platz_index)
            break

    if position is not None:
        break

if position is None:
    print(f"{gesuchter_name} wurde nicht gefunden.")
else:
    print(
        f"{gesuchter_name} befindet sich bei "
        f"Index {position} beziehungsweise in "
        f"Reihe {position[0] + 1}, Platz {position[1] + 1}."
    )

Namen mit A: ['Anna', 'Amir']
Anzahl der Schüler: 20
Flache Liste: ['Anna', 'Ben', 'Cem', 'Daria', 'Elias', 'Amir', 'Gina', 'Hannes', 'Ida', 'Jonas', 'Kira', 'Leon', 'Mert', 'Nina', 'Oskar', 'Pia', 'Quentin', 'Rita', 'Sven', 'Tina']
Mert befindet sich bei Index (2, 2) beziehungsweise in Reihe 3, Platz 3.


### Warum wird zweimal `break` verwendet?

Das erste `break` beendet nur die innere Schleife. Die äußere Schleife würde sonst weiterlaufen. Deshalb wird nach dem Fund zusätzlich geprüft, ob `position` nicht mehr `None` ist.

> **Prüfungsblick:** Für eine benutzerfreundliche Ausgabe werden die Indizes häufig um 1 erhöht, weil Menschen normalerweise ab 1 zählen.

# 7. List Comprehensions

Eine **List Comprehension** ist eine kompakte Schreibweise, mit der aus einer vorhandenen Folge eine neue Liste erzeugt wird.

Grundform:

```python
neue_liste = [ausdruck for element in daten]
```

Mit Filter:

```python
neue_liste = [ausdruck for element in daten if bedingung]
```

Mit `if` und `else`:

```python
neue_liste = [wert_wenn_wahr if bedingung else wert_wenn_falsch
              for element in daten]
```

## 7.1 Punktzahlen als bestanden oder nicht bestanden kennzeichnen

Werte ab 50 gelten als bestanden.

In [ ]:
punkte = [35, 50, 72, 49]

ergebnisse = [
    "bestanden" if punktzahl >= 50 else "nicht bestanden"
    for punktzahl in punkte
]

print("Punkte:    ", punkte)
print("Ergebnisse:", ergebnisse)

Punkte:     [35, 50, 72, 49]
Ergebnisse: ['nicht bestanden', 'bestanden', 'bestanden', 'nicht bestanden']


Der Ausdruck vor dem `for` entscheidet, welcher Text für jeden einzelnen Wert gespeichert wird.

## 7.2 Verschachtelte Liste in eine lineare Liste umwandeln

**Flatten** bedeutet, eine verschachtelte Struktur zu „verflachen“.

Aus:

```python
[[1, 2, 3], [4, 5, 6]]
```

wird:

```python
[1, 2, 3, 4, 5, 6]
```

In [ ]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

flache_liste = [
    zahl
    for zeile in matrix
    for zahl in zeile
]

print(flache_liste)

[1, 2, 3, 4, 5, 6, 7, 8, 9]


Die Reihenfolge entspricht zwei verschachtelten Schleifen:

```python
for zeile in matrix:
    for zahl in zeile:
        ...
```

## 7.3 Wörter nach Länge filtern und umwandeln

Alle Wörter mit mehr als drei Buchstaben sollen ausgewählt und in Großbuchstaben umgewandelt werden.

In [ ]:
words = ["house", "dog", "international", "cup", "AI"]

lange_woerter = [
    wort.upper()
    for wort in words
    if len(wort) > 3
]

print(lange_woerter)

['HOUSE', 'INTERNATIONAL']


Hier übernimmt die List Comprehension zwei Aufgaben:

1. `if len(wort) > 3` filtert die Wörter.
2. `wort.upper()` wandelt die verbleibenden Wörter um.

## 7.4 Zufallszahlen erzeugen und sortieren

Es wird eine Liste mit `n` Zufallszahlen erzeugt und anschließend sortiert.

Dafür wird das Modul `random` benötigt.

In [ ]:
import random

n = 10

zufallszahlen = [
    random.randint(1, 100)
    for _ in range(n)
]

sortierte_zahlen = sorted(zufallszahlen)

print("Unsortiert:", zufallszahlen)
print("Sortiert:  ", sortierte_zahlen)

Unsortiert: [37, 34, 23, 45, 15, 72, 66, 12, 6, 54]
Sortiert:   [6, 12, 15, 23, 34, 37, 45, 54, 66, 72]


Der Unterstrich `_` wird verwendet, weil der konkrete Schleifenwert nicht benötigt wird. Die Schleife soll lediglich `n` Wiederholungen erzeugen.

`sorted()` liefert eine neue sortierte Liste. Die ursprüngliche Liste bleibt erhalten.

## 7.5 Sitzplatzanalyse mit List Comprehensions

In [ ]:
sitzplan_lc = [
    ["Jason", "Barry", "Mark", "Peter", "Andreas", "Maik"],
    ["John", "Jane", "Susi", "Sara", "Sven"],
    ["Max", "Moritz", "Mia", "Anna", "Markus"],
    ["Lukas", "Lena", "Joshua", "Luke", "Oliver"],
]

# Gesamtzahl aller Plätze
gesamtzahl_plaetze = sum(len(reihe) for reihe in sitzplan_lc)

# Anzahl der Plätze je Reihe
plaetze_je_reihe = [len(reihe) for reihe in sitzplan_lc]

# Flache Liste aller Namen
alle_personen = [
    name
    for reihe in sitzplan_lc
    for name in reihe
]

# Namen, die mit M beginnen
namen_mit_m = [
    name
    for reihe in sitzplan_lc
    for name in reihe
    if name.startswith("M")
]

print("Gesamtzahl der Plätze:", gesamtzahl_plaetze)
print("Plätze je Reihe:", plaetze_je_reihe)
print("Alle Personen:", alle_personen)
print("Namen mit M:", namen_mit_m)

Gesamtzahl der Plätze: 21
Plätze je Reihe: [6, 5, 5, 5]
Alle Personen: ['Jason', 'Barry', 'Mark', 'Peter', 'Andreas', 'Maik', 'John', 'Jane', 'Susi', 'Sara', 'Sven', 'Max', 'Moritz', 'Mia', 'Anna', 'Markus', 'Lukas', 'Lena', 'Joshua', 'Luke', 'Oliver']
Namen mit M: ['Mark', 'Maik', 'Max', 'Moritz', 'Mia', 'Markus']


> **Merksatz:** Eine List Comprehension sollte kompakt, aber noch verständlich sein. Bei komplizierten Bedingungen sind normale Schleifen oft leichter zu lesen.

# 8. Lottozahlengenerator – 6 aus 49

## Aufgabenstellung

Es sollen sechs unterschiedliche Lottozahlen aus dem Zahlenbereich 1 bis 49 gezogen werden.

Vorgaben:

- Es muss eine Schleife verwendet werden.
- Keine Zahl darf doppelt vorkommen.
- `random.sample()` darf nicht verwendet werden.

## Lösungsweg

1. Eine leere Liste wird angelegt.
2. Solange weniger als sechs Zahlen gespeichert sind, wird eine Zufallszahl erzeugt.
3. Mit `not in` wird geprüft, ob die Zahl bereits vorhanden ist.
4. Nur neue Zahlen werden hinzugefügt.
5. Für eine übersichtliche Ausgabe wird die Liste anschließend sortiert.

In [ ]:
import random

lottozahlen = []

while len(lottozahlen) < 6:
    gezogene_zahl = random.randint(1, 49)

    if gezogene_zahl not in lottozahlen:
        lottozahlen.append(gezogene_zahl)

lottozahlen.sort()

print("Gezogene Lottozahlen:", lottozahlen)

Gezogene Lottozahlen: [10, 17, 19, 22, 38, 47]


### Warum wird eine `while`-Schleife verwendet?

Vor dem Start ist nicht bekannt, wie viele Zufallsziehungen notwendig sind. Doppelte Zahlen werden verworfen. Die Schleife läuft deshalb so lange, bis tatsächlich sechs unterschiedliche Zahlen vorhanden sind.

> **Prüfungsblick:** `random.randint(1, 49)` kann sowohl 1 als auch 49 erzeugen.

# 9. Figuren mit Schleifen

## 9.1 Umgedrehte Pyramide aus Sternen

Gewünschte Ausgabe:

```text
*******
 *****
  ***
   *
```

Die Anzahl der Sterne sinkt in jedem Durchlauf um zwei. Gleichzeitig steigt die Anzahl der führenden Leerzeichen um eins.

In [ ]:
anzahl_zeilen = 4

for zeile in range(anzahl_zeilen):
    leerzeichen = zeile
    sterne = 7 - 2 * zeile

    print(" " * leerzeichen + "*" * sterne)

*******
 *****
  ***
   *


## Strenge Variante: höchstens ein Stern pro Schreibvorgang

Bei dieser Lösung wird jedes Sternchen einzeln mit `print("*", end="")` ausgegeben.

In [ ]:
anzahl_zeilen = 4

for zeile in range(anzahl_zeilen):
    leerzeichen = zeile
    sterne = 7 - 2 * zeile

    for _ in range(leerzeichen):
        print(" ", end="")

    for _ in range(sterne):
        print("*", end="")

    print()

*******
 *****
  ***
   *


## 9.2 Baum- beziehungsweise Pfeilfigur aus X

Gewünschte Ausgabe:

```text
       X
      XXX
     XXXXX
    XXXXXXX
   XXXXXXXXX
       X
```

Die Krone besteht aus fünf Zeilen. Die Zahl der `X` steigt jeweils um zwei. Anschließend wird der Stamm ausgegeben.

In [ ]:
hoehe = 5
gesamtbreite = 2 * hoehe - 1

for zeile in range(hoehe):
    anzahl_x = 2 * zeile + 1
    leerzeichen = (gesamtbreite - anzahl_x) // 2

    print(" " * leerzeichen + "X" * anzahl_x)

# Stamm mittig unter der Figur
print(" " * (gesamtbreite // 2) + "X")

    X
   XXX
  XXXXX
 XXXXXXX
XXXXXXXXX
    X


### Verwendete Formeln

```python
anzahl_x = 2 * zeile + 1
```

erzeugt die ungeraden Zahlen 1, 3, 5, 7 und 9.

```python
leerzeichen = (gesamtbreite - anzahl_x) // 2
```

berechnet die notwendigen Leerzeichen für eine mittige Ausrichtung.

> **Merksatz:** Figurenaufgaben werden leichter, wenn für jede Zeile getrennt die Zahl der Leerzeichen und Zeichen bestimmt wird.

# 10. Geldbetrag in Scheine und Münzen zerlegen

## Aufgabenstellung

Ein Euro-Betrag soll in möglichst große Scheine und Münzen zerlegt werden.

Beispiel:

```text
Eingabe: 1123,88 Euro
```

Die Stückelungen reichen von 500 Euro bis 1 Cent.

Gesucht sind:

1. ein Struktogramm,
2. ein Programmablaufplan,
3. die Umsetzung in Python.

## Grundidee des Algorithmus

Für jede Stückelung wird berechnet:

```text
Anzahl = Restbetrag ganzzahlig geteilt durch Stückelung
Restbetrag = Restbetrag modulo Stückelung
```

### Wichtig: Berechnung in Cent

Geldbeträge sollten nicht direkt mit Gleitkommazahlen zerlegt werden. Werte wie `0.1` können intern nicht immer exakt dargestellt werden.

Deshalb wird der gesamte Betrag zunächst in Cent umgerechnet:

```python
1123.88 Euro -> 112388 Cent
```

## Vereinfachtes Struktogramm in Textform

```text
┌─────────────────────────────────────────────┐
│ Betrag in Euro festlegen                    │
├─────────────────────────────────────────────┤
│ Betrag in Cent umrechnen                    │
├─────────────────────────────────────────────┤
│ Stückelungen in einer Liste speichern       │
├─────────────────────────────────────────────┤
│ FÜR jede Stückelung                         │
│   ├─ Anzahl = Rest // Stückelung            │
│   ├─ Rest = Rest % Stückelung               │
│   └─ Anzahl und Stückelung ausgeben         │
├─────────────────────────────────────────────┤
│ Ende                                        │
└─────────────────────────────────────────────┘
```

## Programmablaufplan in Textform

```text
Start
  |
Betrag festlegen
  |
Betrag in Cent umrechnen
  |
Stückelungen bereitstellen
  |
Nächste Stückelung vorhanden? ---- Nein ----> Ende
  |
 Ja
  |
Anzahl = Rest // Stückelung
  |
Rest = Rest % Stückelung
  |
Ergebnis ausgeben
  |
Zur nächsten Stückelung
```

In [ ]:
from decimal import Decimal, ROUND_HALF_UP


def geldbetrag_zerlegen(betrag_euro):
    """Zerlegt einen Euro-Betrag in möglichst große Stückelungen."""

    betrag = Decimal(str(betrag_euro))

    if betrag < 0:
        raise ValueError("Der Geldbetrag darf nicht negativ sein.")

    # Genau auf Cent runden und anschließend in eine ganze Centzahl umwandeln.
    betrag = betrag.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    rest_cent = int(betrag * 100)

    stueckelungen_cent = [
        50000, 20000, 10000, 5000, 2000,
        1000, 500, 200, 100, 50,
        20, 10, 5, 2, 1
    ]

    ergebnis = []

    for stueckelung in stueckelungen_cent:
        anzahl = rest_cent // stueckelung
        rest_cent %= stueckelung
        ergebnis.append((stueckelung, anzahl))

    return betrag, ergebnis


def formatiere_stueckelung(stueckelung_cent):
    """Formatiert eine Stückelung als Euro- oder Centangabe."""

    if stueckelung_cent >= 100:
        euro = stueckelung_cent // 100
        return f"{euro} €"

    return f"{stueckelung_cent} Cent"


betrag, zerlegung = geldbetrag_zerlegen("1123.88")

print(f"Geldbetrag: {betrag:.2f} €")
print("-" * 25)

for stueckelung, anzahl in zerlegung:
    print(f"{anzahl:2} × {formatiere_stueckelung(stueckelung)}")

Geldbetrag: 1123.88 €
-------------------------
 2 × 500 €
 0 × 200 €
 1 × 100 €
 0 × 50 €
 1 × 20 €
 0 × 10 €
 0 × 5 €
 1 × 2 €
 1 × 1 €
 1 × 50 Cent
 1 × 20 Cent
 1 × 10 Cent
 1 × 5 Cent
 1 × 2 Cent
 1 × 1 Cent


## Variante ohne Stückelungen mit Anzahl 0

In der Praxis ist häufig nur interessant, welche Scheine und Münzen tatsächlich benötigt werden.

In [ ]:
print(f"Geldbetrag: {betrag:.2f} €")
print("-" * 25)

for stueckelung, anzahl in zerlegung:
    if anzahl > 0:
        print(f"{anzahl:2} × {formatiere_stueckelung(stueckelung)}")

Geldbetrag: 1123.88 €
-------------------------
 2 × 500 €
 1 × 100 €
 1 × 20 €
 1 × 2 €
 1 × 1 €
 1 × 50 Cent
 1 × 20 Cent
 1 × 10 Cent
 1 × 5 Cent
 1 × 2 Cent
 1 × 1 Cent


## Erklärung der beiden wichtigsten Operatoren

### Ganzzahlige Division `//`

```python
112388 // 50000
```

ergibt `2`, weil zwei 500-Euro-Scheine vollständig in den Betrag passen.

### Modulo `%`

```python
112388 % 50000
```

ergibt den Restbetrag, der nach den beiden 500-Euro-Scheinen noch zerlegt werden muss.

> **Prüfungsblick:** Die Stückelungen müssen absteigend sortiert sein. Nur dann werden zuerst die größtmöglichen Scheine und Münzen verwendet.

# Zusammenfassung

In diesem Notebook wurden mehrere wichtige Python-Grundlagen praktisch angewendet:

- `if`, `elif` und `else`
- logische Operatoren wie `and`, `or` und `not`
- Entscheidungstabellen
- Wahrheitstabellen und Logikgatter
- `for`- und `while`-Schleifen
- verschachtelte Schleifen
- Listen und verschachtelte Listen
- `enumerate()`
- List Comprehensions
- Zufallszahlen mit `random`
- Funktionen und Rückgabewerte
- Ganzzahldivision und Modulo
- sichere Verarbeitung von Geldbeträgen mit `Decimal`

## Abschließender Kurz-Check

1. Weshalb hat bei der Parkhausaufgabe die Sperrung Vorrang?
2. Weshalb bekommt ein minderjähriger VIP keinen Sekt?
3. Welche Wahrheitstabelle besitzt ein XOR-Gatter?
4. Was bedeutet „Flatten“?
5. Weshalb eignet sich eine `while`-Schleife für den Lottozahlengenerator?
6. Weshalb wird der Geldbetrag in Cent umgerechnet?
7. Was ist der Unterschied zwischen `//` und `%`?

Die Antworten lassen sich direkt aus den jeweiligen Kapiteln herleiten.